# Dia 10 — Scraping (raspagem de páginas)

**🃏 Cartas:** 10 · 31 · 52 &nbsp;·&nbsp; **📘 Original:** *Day 10. Scraping*
&nbsp;·&nbsp; [repositório](https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias) &nbsp;·&nbsp; [curso original do cipher387](https://github.com/cipher387/python-for-OSINT-21-days)

---

## 📖 Capítulo 10 — A vitrine congelada

Os buscadores e os Diários Oficiais de ontem trouxeram nomes, mas nenhum endereço de site no ar.
A MegaOferta Brasil sumiu. A Iara não parece preocupada: ela abre um pendrive etiquetado
**"vitrine — cópia de 29/11"**.

— Uma das vítimas salvou a página no navegador antes de pagar. Para ela, era só um print caprichado.
Para nós, é o código-fonte inteiro da loja.

Ela abre o arquivo no navegador. Você vê iPhones, PlayStations, um selo "Só no PIX". Depois ela aperta
**Ctrl+U** e a tela vira uma sopa de tags.

— O cliente vê a vitrine. Nós lemos os bastidores. Programador apressado deixa bilhete no código,
e plataforma de loja pronta assina o próprio nome no cabeçalho.

Ela empurra o teclado para você.

— Quero três coisas: a lista de produtos com preço, tudo o que identifica quem recebe o dinheiro, e
qualquer coisa que o navegador **não mostra**. E nada de instalar biblioteca: o Python já traz um
leitor de HTML. Se um dia precisarmos visitar um site de verdade, primeiro perguntamos ao `robots.txt`
o que ele permite.

Você respira fundo e escreve `from html.parser import HTMLParser`.

In [ ]:
#@title ⚙️ Preparar o ambiente (rode esta célula primeiro)
# No Colab: baixa o repositório e entra na pasta do dia.
# No Jupyter local: o notebook já está na pasta do dia, então não faz nada.
import os, subprocess, sys
from pathlib import Path

REPO = "https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias"
if "google.colab" in sys.modules:
    destino = "/content/python-para-osint-21-dias"
    if not os.path.isdir(destino):
        subprocess.run(["git", "clone", "--depth", "1", REPO, destino], check=True)
    os.chdir(f"{destino}/dia-10-scraping")

PASTA_NB = str(Path.cwd())          # usada pelas células para voltar sempre à pasta certa
print("📂 Pasta do dia:", PASTA_NB)
print("🐍 Python", sys.version.split()[0])

## 🎯 Objetivos

- Rodar o exemplo original com `requests` + `BeautifulSoup`.
- Entender o HTML como uma árvore de **tags**, **atributos** e **textos**.
- Raspar uma página salva com `html.parser`, sem instalar nada.
- Achar o que o navegador esconde: **comentários** e **metadados** (`<meta name="generator">`).
- Consultar o `robots.txt` com `urllib.robotparser` antes de visitar um site real.

## 🧠 Conceito em 2 minutos

**Raspar** (*scraping*) é ler o HTML de uma página e tirar dele só o que interessa. Um trecho de HTML
tem três ingredientes:

```html
<span class="preco">R$ 1.999,00</span>
 └tag┘ └─atributo─┘ └──texto──┘
```

Para achar elementos, usamos **seletores** parecidos com os do CSS:

| Seletor | Significa |
|---|---|
| `h1` | todo elemento `<h1>` |
| `.preco` | elementos com `class="preco"` |
| `#pagamento` | o elemento com `id="pagamento"` |
| `div.produto` | `<div>` com a classe `produto` |

O **BeautifulSoup** monta a árvore inteira e deixa você perguntar `soup.find("h1")`. O
**`html.parser`** da biblioteca padrão é mais "baixo nível": ele lê o arquivo do começo ao fim e avisa
você a cada tag que abre, cada texto e cada tag que fecha. Dá mais trabalho, mas não precisa instalar
nada e ensina como a coisa funciona por dentro.

Duas armadilhas comuns: (1) muitos sites montam o conteúdo com **JavaScript** depois de carregar, e
o Python só recebe o "esqueleto" (aí entram ferramentas como Selenium ou Playwright); (2) antes de
escrever um raspador, procure se já existe ferramenta pronta para aquele site.

## 🥋 Nível 1 — Exercício original (cipher387)

| Arquivo | O que faz |
|---|---|
| [`original/scraping.py`](https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias/blob/main/dia-10-scraping/original/scraping.py) | Baixa a página do pacote `duckduckgo-search` no PyPI e imprime o título `<h1>` com BeautifulSoup |

```bash
pip install requests beautifulsoup4
cd dia-10-scraping
python original/scraping.py
```

> 💡 Troque a URL por outra página pública e mude `"h1"` para `"title"` ou `"h2"`. Depois experimente
> `soup.find_all("a")` para listar todos os links.

In [ ]:
#@title 📦 Instalar as bibliotecas do exercício original
%pip install -q requests beautifulsoup4

### `original/scraping.py` — Raspar o título (h1) de uma página do PyPI

In [ ]:
%cd -q {PASTA_NB}/original
# ──────────────────────────────────────────────────────────────────────────
# Título: Raspar o título (h1) de uma página do PyPI
# Código ORIGINAL de cipher387 (Cyb_detective), "Python for OSINT. 21-day course"
# Fonte: https://github.com/cipher387/python-for-OSINT-21-days/blob/main/Day_10/scraping.py
# Licença: MIT © 2023 Cyb_detective. O código está intacto; só foram
# acrescentados comentários em pt-BR (linhas que começam com "# 💬").
# Requer: pip install requests beautifulsoup4
# ──────────────────────────────────────────────────────────────────────────

# 💬 requests baixa a página; BeautifulSoup (pacote beautifulsoup4) entende o HTML.
import requests
from bs4 import BeautifulSoup

# 💬 A página do PyPI de um pacote. Troque por outra página pública para testar.
url = "https://pypi.org/project/duckduckgo-search/"

# 💬 Faz o pedido HTTP GET. A resposta traz o HTML bruto em web_page.content.
web_page = requests.get(url)

# 💬 Cria a "sopa": o HTML vira uma árvore de elementos que dá para pesquisar.
# 💬 "html.parser" é o analisador que já vem com o Python (o lxml é mais rápido, mas precisa instalar).
soup = BeautifulSoup(web_page.content, "html.parser")

# 💬 find("h1") devolve o PRIMEIRO elemento <h1>; get_text() tira as tags e deixa só o texto.
# 💬 Se a página não tiver <h1>, find devolve None e get_text() dá AttributeError.
header = soup.find("h1").get_text()

# 💬 Mostra o título encontrado.
print(header)

## 🇧🇷 Nível 2 — Camada Brasil

Tudo com a **biblioteca padrão**. Os dois primeiros são 100% offline (cópia salva da loja fictícia).

| Arquivo | O que faz |
|---|---|
| [`br/01_html_parser_vitrine.py`](https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias/blob/main/dia-10-scraping/br/01_html_parser_vitrine.py) | Extrai da vitrine: produtos, preços (em `Decimal`), chave PIX, CNPJ e links; salva JSON |
| [`br/02_comentarios_e_metadados.py`](https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias/blob/main/dia-10-scraping/br/02_comentarios_e_metadados.py) | Pega `<title>`, `<meta>` e os **comentários escondidos** com `handle_comment` |
| [`br/03_raspar_com_etica.py`](https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias/blob/main/dia-10-scraping/br/03_raspar_com_etica.py) | Lê o `robots.txt`, espera, baixa `https://www.gov.br/pt-br` e lista título e links |

```bash
cd dia-10-scraping
python br/01_html_parser_vitrine.py
python br/02_comentarios_e_metadados.py
OSINT_OFFLINE=1 python br/03_raspar_com_etica.py   # força o modo offline
```

**O que você deve ver no caso:** 6 produtos "Só no PIX", a chave `f38b2ffc-…`, o CNPJ inválido do
rodapé e dois comentários esquecidos pelo desenvolvedor: um **TODO para trocar a chave PIX antes da
Black Friday** e o e-mail **`kau.ofertas@example.com`**. O `generator` é **LojaPronta 3.2**.

> 💡 Os campos com formulário à direita (`#@param`) podem ser editados sem mexer no código. Para testar sem internet, rode antes: `os.environ['OSINT_OFFLINE'] = '1'`.

### `br/01_html_parser_vitrine.py` — Raspar a vitrine congelada da loja falsa só com a biblioteca padrão (html.parser).

In [ ]:
%cd -q {PASTA_NB}
"""
Raspar a vitrine congelada da loja falsa só com a biblioteca padrão (html.parser).

Carta 31 (rank B) — Jutsu da Vitrine Congelada

O que você aprende:
  • que um HTML é uma sequência de "eventos": abre tag, texto, fecha tag;
  • a usar html.parser.HTMLParser (nativo) no lugar do BeautifulSoup;
  • a "lembrar onde estou" (dentro de um produto? dentro do preço?) com variáveis;
  • a extrair produtos, preços, chave PIX, CNPJ e links da cópia salva do site.

Este script é 100% offline: lê caso/dados/vitrine_megaoferta.html (fictício).

Como rodar (dentro da pasta dia-10-scraping):
    python br/01_html_parser_vitrine.py
"""

import json                          # para salvar o resultado organizado
from decimal import Decimal          # dinheiro sem erro de arredondamento
from html.parser import HTMLParser   # o "BeautifulSoup" de bolso que vem com o Python
from pathlib import Path

# ── Parâmetros (no Colab viram um formulário) ─────────────────────────────
ARQUIVO_HTML = "vitrine_megaoferta.html"  #@param {type:"string"}

PASTA_DIA = Path(__file__).resolve().parent.parent if "__file__" in globals() else Path.cwd()
PASTA_CASO = PASTA_DIA.parent / "caso" / "dados"


class LeitorDeVitrine(HTMLParser):
    """
    O HTMLParser lê o HTML do começo ao fim e chama um método a cada "evento":
      handle_starttag → achou <tag ...>
      handle_data     → achou um texto entre tags
      handle_endtag   → achou </tag>
    Nós só escrevemos o que fazer em cada caso.
    """

    def __init__(self):
        super().__init__()
        self.produtos = []        # lista de dicionários {"nome": ..., "preco": ...}
        self.links = []           # todos os href encontrados
        self.chave_pix = None
        self.textos_rodape = []   # textos soltos do <footer> (onde costuma estar o CNPJ)
        # "Onde estou agora?" Começa tudo como False/None.
        self.no_produto = False
        self.campo_atual = None   # "nome", "preco", "pix" ou None
        self.no_rodape = False

    def handle_starttag(self, tag, attrs):
        # attrs chega como lista de pares [("class", "produto")]; dict() facilita a leitura.
        atributos = dict(attrs)
        classe = atributos.get("class", "")

        if tag == "div" and classe == "produto":
            self.no_produto = True
            self.produtos.append({"nome": "", "preco": ""})   # novo produto, ainda vazio
        elif self.no_produto and tag == "h2":
            self.campo_atual = "nome"
        elif self.no_produto and tag == "span" and classe == "preco":
            self.campo_atual = "preco"
        elif tag == "strong" and classe == "pix":
            self.campo_atual = "pix"
        elif tag == "footer":
            self.no_rodape = True

        # Qualquer tag com href (links <a>) entra na lista de links.
        if "href" in atributos:
            self.links.append(atributos["href"])

    def handle_data(self, texto):
        texto = texto.strip()
        if not texto:
            return                       # ignora espaços e quebras de linha entre as tags
        if self.campo_atual == "nome":
            self.produtos[-1]["nome"] += texto    # [-1] = o último produto criado
        elif self.campo_atual == "preco":
            self.produtos[-1]["preco"] += texto
        elif self.campo_atual == "pix":
            self.chave_pix = texto
        elif self.no_rodape:
            self.textos_rodape.append(texto)

    def handle_endtag(self, tag):
        # Ao fechar a tag, saímos do "campo" em que estávamos.
        if tag in ("h2", "span", "strong"):
            self.campo_atual = None
        elif tag == "div":
            self.no_produto = False
        elif tag == "footer":
            self.no_rodape = False


def preco_para_decimal(texto):
    """'R$ 1.999,00' → Decimal('1999.00'). Tira 'R$', pontos de milhar e troca vírgula por ponto."""
    limpo = texto.replace("R$", "").replace(".", "").replace(",", ".").strip()
    return Decimal(limpo)


# ── 1. Ler o arquivo e alimentar o leitor ─────────────────────────────────
html = (PASTA_CASO / ARQUIVO_HTML).read_text(encoding="utf-8")
leitor = LeitorDeVitrine()
leitor.feed(html)          # feed() dispara todos os handle_* do começo ao fim

# ── 2. Produtos e preços ──────────────────────────────────────────────────
print(f"🛒 Produtos na vitrine: {len(leitor.produtos)}\n")
for produto in leitor.produtos:
    produto["valor"] = str(preco_para_decimal(produto["preco"]))   # guardamos como texto no JSON
    print(f"  • {produto['nome']:<22} {produto['preco']:>12}")

# ── 3. Chave PIX, CNPJ e links ────────────────────────────────────────────
print(f"\n💸 Chave PIX exibida: {leitor.chave_pix}")

cnpj = None
for texto in leitor.textos_rodape:
    if "CNPJ" in texto:
        # "MegaOferta ... LTDA — CNPJ 48.217.093/0001-30" → pega o que vem depois de "CNPJ"
        cnpj = texto.split("CNPJ", 1)[1].strip()
print(f"🏢 CNPJ no rodapé:    {cnpj}  (no Dia 2 você viu que ele NÃO passa no dígito verificador)")

print("\n🔗 Links encontrados:")
for link in leitor.links:
    # Classificamos o tipo de link pelo começo do endereço.
    if link.startswith("mailto:"):
        tipo = "e-mail"
    elif "wa.me" in link:
        tipo = "WhatsApp"
    else:
        tipo = "página"
    print(f"  [{tipo:<8}] {link}")

# ── 4. Salvar tudo em JSON para os próximos dias ──────────────────────────
saidas = PASTA_DIA / "saidas"
saidas.mkdir(exist_ok=True)
resultado = {"produtos": leitor.produtos, "chave_pix": leitor.chave_pix, "cnpj": cnpj, "links": leitor.links}
(saidas / "vitrine_extraida.json").write_text(json.dumps(resultado, indent=2, ensure_ascii=False), encoding="utf-8")
print("\n💾 saidas/vitrine_extraida.json gravado.")

### `br/02_comentarios_e_metadados.py` — O que o navegador esconde: comentários HTML e metadados da vitrine.

In [ ]:
%cd -q {PASTA_NB}
"""
O que o navegador esconde: comentários HTML e metadados da vitrine.

Carta 31 (rank B) — Jutsu da Vitrine Congelada (parte 2)

O que você aprende:
  • que comentários <!-- ... --> não aparecem na tela, mas estão no código;
  • a capturá-los com handle_comment;
  • a ler as tags <meta> (description, generator) e o <title>;
  • por que o "generator" funciona como impressão digital da plataforma da loja.

Este script é 100% offline: lê caso/dados/vitrine_megaoferta.html (fictício).

Como rodar (dentro da pasta dia-10-scraping):
    python br/02_comentarios_e_metadados.py
"""

from html.parser import HTMLParser
from pathlib import Path

ARQUIVO_HTML = "vitrine_megaoferta.html"  #@param {type:"string"}

PASTA_DIA = Path(__file__).resolve().parent.parent if "__file__" in globals() else Path.cwd()
PASTA_CASO = PASTA_DIA.parent / "caso" / "dados"


class LeitorDeBastidores(HTMLParser):
    """Guarda comentários, metadados e o título da página."""

    def __init__(self):
        super().__init__()
        self.comentarios = []
        self.metas = {}          # {"generator": "LojaPronta 3.2", ...}
        self.titulo = ""
        self.no_titulo = False

    def handle_comment(self, texto):
        # Chamado para cada <!-- ... -->. "texto" é o que está entre <!-- e -->.
        self.comentarios.append(texto.strip())

    def handle_starttag(self, tag, attrs):
        atributos = dict(attrs)
        if tag == "meta" and "name" in atributos:
            # <meta name="generator" content="LojaPronta 3.2">
            self.metas[atributos["name"]] = atributos.get("content", "")
        elif tag == "meta" and "charset" in atributos:
            self.metas["charset"] = atributos["charset"]
        elif tag == "title":
            self.no_titulo = True

    def handle_data(self, texto):
        if self.no_titulo:
            self.titulo += texto

    def handle_endtag(self, tag):
        if tag == "title":
            self.no_titulo = False


html = (PASTA_CASO / ARQUIVO_HTML).read_text(encoding="utf-8")
leitor = LeitorDeBastidores()
leitor.feed(html)

# ── 1. Título e metadados ─────────────────────────────────────────────────
print(f"📰 Título: {leitor.titulo.strip()}\n")
print("🏷️  Metadados:")
for nome, valor in leitor.metas.items():
    print(f"   {nome:<12} = {valor}")

# ── 2. Comentários escondidos ─────────────────────────────────────────────
print(f"\n🙈 Comentários escondidos no HTML: {len(leitor.comentarios)}")
for comentario in leitor.comentarios:
    print(f"   <!-- {comentario} -->")

# ── 3. O que isso significa para o caso ───────────────────────────────────
print("\n🧠 Leitura de analista:")
for comentario in leitor.comentarios:
    texto = comentario.lower()
    if "pix" in texto:
        print("   🚩 Um TODO fala em trocar a chave PIX antes da Black Friday:")
        print("      indica planejamento e que pode haver MAIS de uma chave em uso.")
    if "@" in texto:
        email = [p for p in comentario.split() if "@" in p][0]
        print(f"   🚩 E-mail do desenvolvedor esquecido no código: {email}")
        print("      Compare com o apelido do vendedor no WhatsApp ('Kau').")

gerador = leitor.metas.get("generator")
if gerador:
    print(f"   🧬 Plataforma declarada: '{gerador}'. Outras lojas falsas feitas na mesma")
    print("      plataforma/versão costumam repetir o mesmo 'generator', o mesmo layout e")
    print("      os mesmos erros. É uma impressão digital para achar sites irmãos.")

### `br/03_raspar_com_etica.py` — Raspar com educação: consultar o robots.txt antes de baixar uma página pública.

In [ ]:
%cd -q {PASTA_NB}
"""
Raspar com educação: consultar o robots.txt antes de baixar uma página pública.

Carta 31 (rank B) — Jutsu da Vitrine Congelada (parte 3)

O que você aprende:
  • o que é o robots.txt e como lê-lo com urllib.robotparser (nativo);
  • a se identificar com um User-Agent honesto e a esperar entre pedidos;
  • a baixar uma página pública e extrair o <title> e os links com html.parser;
  • a transformar links relativos ("/pt-br/servicos") em absolutos com urljoin.

Alvo padrão: o portal https://www.gov.br/pt-br (página pública institucional).

Como rodar (dentro da pasta dia-10-scraping):
    python br/03_raspar_com_etica.py
    OSINT_OFFLINE=1 python br/03_raspar_com_etica.py   # usa as amostras de dados/
"""

import os
import time                          # para esperar entre um pedido e outro
import urllib.parse                  # urlsplit e urljoin
import urllib.request
import urllib.robotparser            # o "leitor de robots.txt" do Python
from html.parser import HTMLParser
from pathlib import Path

# ── Parâmetros ────────────────────────────────────────────────────────────
URL = "https://www.gov.br/pt-br"  #@param {type:"string"}
USER_AGENT = "python-para-osint-21-dias/estudo"  #@param {type:"string"}
PAUSA_SEGUNDOS = 2  #@param {type:"integer"}
MAX_LINKS = 15  #@param {type:"integer"}

PASTA_DIA = Path(__file__).resolve().parent.parent if "__file__" in globals() else Path.cwd()
OFFLINE = os.environ.get("OSINT_OFFLINE") == "1"


def baixar_texto(url, amostra):
    """Baixa uma página como texto; se falhar (ou OFFLINE=1), lê a amostra local."""
    if not OFFLINE:
        try:
            pedido = urllib.request.Request(url, headers={"User-Agent": USER_AGENT})
            with urllib.request.urlopen(pedido, timeout=15) as resposta:
                # O servidor diz a codificação no cabeçalho; se não disser, assumimos utf-8.
                codificacao = resposta.headers.get_content_charset() or "utf-8"
                return resposta.read().decode(codificacao, errors="replace")
        except Exception as erro:
            print(f"⚠️ MODO OFFLINE: a consulta falhou ({erro}). Usando a amostra local.")
    else:
        print("⚠️ MODO OFFLINE: usando a amostra local.")
    return (PASTA_DIA / "dados" / amostra).read_text(encoding="utf-8")


class LeitorDeLinks(HTMLParser):
    """Pega o <title> e todos os <a href> da página."""

    def __init__(self):
        super().__init__()
        self.titulo = ""
        self.no_titulo = False
        self.links = []

    def handle_starttag(self, tag, attrs):
        if tag == "title":
            self.no_titulo = True
        elif tag == "a":
            href = dict(attrs).get("href")
            if href:
                self.links.append(href)

    def handle_data(self, texto):
        if self.no_titulo:
            self.titulo += texto

    def handle_endtag(self, tag):
        if tag == "title":
            self.no_titulo = False


# ── 1. Ler o robots.txt do site ───────────────────────────────────────────
# O robots.txt fica SEMPRE na raiz: https://www.gov.br/robots.txt
partes = urllib.parse.urlsplit(URL)
url_robots = f"{partes.scheme}://{partes.netloc}/robots.txt"
print(f"🤖 Lendo {url_robots}")

texto_robots = baixar_texto(url_robots, "robots_gov_br.txt")
regras = urllib.robotparser.RobotFileParser()
regras.parse(texto_robots.splitlines())     # parse() aceita a lista de linhas que já baixamos

permitido = regras.can_fetch(USER_AGENT, URL)
print(f"   Posso baixar {URL}?  {'✅ sim' if permitido else '⛔ não'}")
atraso = regras.crawl_delay(USER_AGENT)     # alguns sites pedem um intervalo mínimo
if atraso:
    print(f"   O site pede {atraso} s entre pedidos (Crawl-delay). Vamos respeitar.")
    PAUSA_SEGUNDOS = max(PAUSA_SEGUNDOS, int(atraso))

if not permitido:
    raise SystemExit("⛔ O robots.txt pede para robôs não acessarem esta página. Paramos aqui.")

# ── 2. Esperar um pouco e baixar a página ─────────────────────────────────
if not OFFLINE:
    time.sleep(PAUSA_SEGUNDOS)   # educação: não emende pedidos em sequência
html = baixar_texto(URL, "gov_br_pt-br.html")

# ── 3. Extrair título e links ─────────────────────────────────────────────
leitor = LeitorDeLinks()
leitor.feed(html)
print(f"\n📰 Título: {' '.join(leitor.titulo.split())}")

# urljoin resolve links relativos: "/pt-br/servicos" + URL → "https://www.gov.br/pt-br/servicos"
absolutos = []
for link in leitor.links:
    completo = urllib.parse.urljoin(URL, link)
    if completo.startswith("http") and completo not in absolutos:   # sem repetidos, sem "javascript:"
        absolutos.append(completo)

internos = [l for l in absolutos if urllib.parse.urlsplit(l).netloc == partes.netloc]
print(f"🔗 {len(absolutos)} links únicos ({len(internos)} do próprio site). Primeiros {MAX_LINKS}:")
for link in absolutos[:MAX_LINKS]:
    marca = "🏠" if link in internos else "🌍"
    print(f"   {marca} {link}")

# ── 4. Guardar a lista (com data/hora da coleta, boa prática de registro) ─
saidas = PASTA_DIA / "saidas"
saidas.mkdir(exist_ok=True)
carimbo = time.strftime("%Y-%m-%d %H:%M:%S")
with open(saidas / "links_coletados.txt", "w", encoding="utf-8") as arquivo:
    arquivo.write(f"# Coleta de {URL} em {carimbo} (hora local) — UA: {USER_AGENT}\n")
    arquivo.write("\n".join(absolutos) + "\n")
print("\n💾 saidas/links_coletados.txt gravado.")

## ⚖️ Ética, LGPD e OPSEC

- **Página salva é a melhor fonte.** Analisar uma cópia (como a da vítima) não deixa rastro no
  servidor do suspeito. Visitar o site ao vivo deixa seu IP no log dele e pode alertá-lo (Dia 12).
- **`robots.txt` não é lei, mas é a vontade declarada do dono do site.** Respeite-o, junto com os
  **termos de uso**. Contornar bloqueio técnico (captcha, login, limite) pode configurar ilícito.
- **Ritmo humano:** pausa entre pedidos e User-Agent honesto. Raspar rápido demais derruba sites
  pequenos e dá bloqueio de IP.
- **Raspou dado pessoal? Aplica-se a LGPD.** Colete só o necessário para a finalidade, guarde com
  segurança e registre data/hora e URL de cada coleta (o script 03 já faz isso).
- O fato de uma página ser pública não autoriza montar bases em massa de pessoas (ex.: raspar perfis
  de uma rede social inteira).

## 🎯 Quiz rápido

Escolha a resposta no formulário de cada célula e rode a célula.

In [ ]:
#@title 1. No html.parser, qual método recebe o conteúdo de um <!-- comentário -->?
resposta = "Escolha uma opção" #@param ["Escolha uma opção", "A) handle_starttag", "B) handle_data", "C) handle_comment", "D) handle_endtag"]
correta = "C) handle_comment"
if resposta == "Escolha uma opção":
    print("👆 Escolha uma opção no formulário e rode de novo.")
elif resposta == correta:
    print("✅ Acertou!", "handle_comment é chamado para cada comentário HTML; ele não aparece na tela, mas está no código.")
else:
    print("❌ Ainda não. Releia o conceito do dia e tente outra vez.")

In [ ]:
#@title 2. O que a tag <meta name='generator' content='LojaPronta 3.2'> revela?
resposta = "Escolha uma opção" #@param ["Escolha uma opção", "A) O IP do servidor", "B) A plataforma (e versão) usada para montar o site", "C) O dono do domínio", "D) A chave PIX da loja"]
correta = "B) A plataforma (e versão) usada para montar o site"
if resposta == "Escolha uma opção":
    print("👆 Escolha uma opção no formulário e rode de novo.")
elif resposta == correta:
    print("✅ Acertou!", "O generator é a assinatura da ferramenta que gerou a página; serve de impressão digital para achar sites irmãos.")
else:
    print("❌ Ainda não. Releia o conceito do dia e tente outra vez.")

In [ ]:
#@title 3. Antes de raspar um site ao vivo, o script consulta o robots.txt para…
resposta = "Escolha uma opção" #@param ["Escolha uma opção", "A) descobrir a senha do painel", "B) saber o que o dono do site pede que robôs não acessem", "C) ficar anônimo", "D) baixar o site mais rápido"]
correta = "B) saber o que o dono do site pede que robôs não acessem"
if resposta == "Escolha uma opção":
    print("👆 Escolha uma opção no formulário e rode de novo.")
elif resposta == correta:
    print("✅ Acertou!", "O robots.txt declara as regras para robôs; urllib.robotparser.can_fetch responde se a URL é permitida.")
else:
    print("❌ Ainda não. Releia o conceito do dia e tente outra vez.")

# 🏆 Desafio S — Dia 10: Caçador de lojas irmãs

**Carta 52 · Rank S**

## A missão

Golpista raramente monta uma loja só. Quando uma cai, outra sobe com outro nome, mas costuma carregar
os mesmos "vícios": a mesma plataforma, a mesma chave PIX, o mesmo WhatsApp, o mesmo e-mail esquecido
num comentário. A Iara recebeu de outras delegacias três páginas salvas (fictícias) em
[`dados/lojas_suspeitas/`](https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias/blob/main/dia-10-scraping/dados/lojas_suspeitas). Quais delas são irmãs da MegaOferta?

Escreva um script que:

1. Extraia a **impressão digital** da vitrine do caso (`caso/dados/vitrine_megaoferta.html`):
   `generator`, chaves PIX, números de `wa.me/`, e-mails de `mailto:` **e de comentários**.
2. Extraia a mesma impressão digital de cada `.html` da pasta de suspeitas.
3. Dê **pontos** por coincidência, com pesos diferentes (plataforma vale pouco, PIX e telefone valem muito).
4. Marque com 🚩 quem passar de `PONTOS_PARA_ALERTA` (padrão: 3).
5. Grave `saidas/lojas_irmas.csv` com `;` e `utf-8-sig`.

## Critérios de rank S

- [ ] Só biblioteca padrão (`html.parser`, `csv`, `pathlib`)
- [ ] Usa **conjuntos** (`set`) e interseção (`&`) para comparar traços
- [ ] Uma página ilegível não derruba o script
- [ ] A loja sem relação com o caso fica com 0 pontos
- [ ] Explica em um comentário por que `generator` sozinho **não** prova nada

## Dica

Monte um dicionário `{"generator": set(), "pix": set(), ...}` dentro do seu `HTMLParser`. Comparar duas
lojas vira `referencia["pix"] & suspeita["pix"]`.

In [ ]:
%cd -q {PASTA_NB}
# ✍️ Escreva sua solução aqui.

In [ ]:
#@title 🔒 Gabarito (dê dois cliques aqui para ver o código) { display-mode: "form" }
%cd -q {PASTA_NB}
"""
Gabarito do Desafio S do Dia 10 — Caçador de lojas irmãs.

Extrai a "impressão digital" da vitrine do caso (generator, chaves PIX, números de
WhatsApp, e-mails escondidos em comentários) e compara com cada página salva em
dados/lojas_suspeitas/. Gera um placar de semelhança e grava saidas/lojas_irmas.csv.
100% offline: só arquivos locais e fictícios.
"""

import csv
from html.parser import HTMLParser
from pathlib import Path

PASTA_SUSPEITAS = "dados/lojas_suspeitas"  #@param {type:"string"}
PONTOS_PARA_ALERTA = 3  #@param {type:"integer"}

PASTA_DIA = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
PASTA_CASO = PASTA_DIA.parent / "caso" / "dados"

# Quanto vale cada coincidência. Chave PIX e telefone pesam mais que a plataforma,
# porque muitas lojas honestas usam a mesma plataforma.
PESOS = {"generator": 1, "pix": 3, "whatsapp": 3, "email": 2}


class Digital(HTMLParser):
    """Coleta os traços que identificam quem montou a loja."""

    def __init__(self):
        super().__init__()
        self.tracos = {"generator": set(), "pix": set(), "whatsapp": set(), "email": set()}
        self.titulo = ""
        self._no_titulo = False
        self._no_pix = False

    def handle_starttag(self, tag, attrs):
        a = dict(attrs)
        if tag == "meta" and a.get("name") == "generator":
            self.tracos["generator"].add(a.get("content", ""))
        if tag == "strong" and a.get("class") == "pix":
            self._no_pix = True
        if tag == "title":
            self._no_titulo = True
        href = a.get("href", "")
        if "wa.me/" in href:
            self.tracos["whatsapp"].add(href.rsplit("/", 1)[-1])     # só os dígitos
        if href.startswith("mailto:"):
            self.tracos["email"].add(href[len("mailto:"):].lower())

    def handle_endtag(self, tag):
        if tag == "strong":
            self._no_pix = False
        if tag == "title":
            self._no_titulo = False

    def handle_data(self, texto):
        if self._no_pix and texto.strip():
            self.tracos["pix"].add(texto.strip().lower())
        if self._no_titulo:
            self.titulo += texto.strip()

    def handle_comment(self, texto):
        for palavra in texto.split():
            if "@" in palavra:
                self.tracos["email"].add(palavra.strip(".,;").lower())


def digital_de(caminho):
    leitor = Digital()
    leitor.feed(caminho.read_text(encoding="utf-8"))
    return leitor


referencia = digital_de(PASTA_CASO / "vitrine_megaoferta.html")
print(f"🧬 Digital da referência ({referencia.titulo}):")
for tipo, valores in referencia.tracos.items():
    print(f"   {tipo:<9} {', '.join(sorted(valores)) or '—'}")

linhas = []
for pagina in sorted((PASTA_DIA / PASTA_SUSPEITAS).glob("*.html")):
    try:
        suspeita = digital_de(pagina)
    except (OSError, UnicodeDecodeError) as erro:
        print(f"⚠️  Não consegui ler {pagina.name}: {erro}")
        continue
    pontos, coincidencias = 0, []
    for tipo, peso in PESOS.items():
        comuns = referencia.tracos[tipo] & suspeita.tracos[tipo]     # interseção de conjuntos
        if comuns:
            pontos += peso
            coincidencias.append(f"{tipo}={'/'.join(sorted(comuns))}")
    alerta = "🚩 provável loja irmã" if pontos >= PONTOS_PARA_ALERTA else "ok"
    linhas.append({"arquivo": pagina.name, "titulo": suspeita.titulo, "pontos": pontos,
                   "coincidencias": " | ".join(coincidencias), "alerta": alerta})

linhas.sort(key=lambda l: l["pontos"], reverse=True)
print("\n🏁 Placar:")
for l in linhas:
    print(f"   {l['pontos']:>2} pts  {l['arquivo']:<36} {l['alerta']}")
    if l["coincidencias"]:
        print(f"           ↳ {l['coincidencias']}")

saidas = PASTA_DIA / "saidas"
saidas.mkdir(exist_ok=True)
with open(saidas / "lojas_irmas.csv", "w", newline="", encoding="utf-8-sig") as f:
    escritor = csv.DictWriter(f, fieldnames=["arquivo", "titulo", "pontos", "coincidencias", "alerta"], delimiter=";")
    escritor.writeheader()
    escritor.writerows(linhas)
print("\n💾 saidas/lojas_irmas.csv gravado.")

## 🔎 Inovação OSINT do dia

**Extrair só o texto principal com `trafilatura`.** Em vez de escrever um seletor para cada site, a
biblioteca Python `trafilatura` tenta separar automaticamente o conteúdo principal (texto do artigo,
título, data, autor) do "lixo" da página (menus, rodapés, anúncios). É muito usada para montar bases de
notícias. Uso básico:

```python
# pip install trafilatura
import trafilatura
html = trafilatura.fetch_url("https://www.gov.br/pt-br")
print(trafilatura.extract(html))
```

Ótima para monitorar notícias sobre golpes; fraca para lojas e páginas com pouco texto corrido.

## 📚 Para ir além

- `html.parser` (documentação oficial): <https://docs.python.org/pt-br/3/library/html.parser.html>
- `urllib.robotparser`: <https://docs.python.org/pt-br/3/library/urllib.robotparser.html>
- Beautiful Soup: <https://www.crummy.com/software/BeautifulSoup/bs4/doc/>
- Protocolo de exclusão de robôs (RFC 9309): <https://www.rfc-editor.org/rfc/rfc9309>
- trafilatura: <https://trafilatura.readthedocs.io/>

## ✅ Checklist

- [ ] Rodei o script original e troquei a URL/seletor
- [ ] Listei os produtos e preços da vitrine com `html.parser`
- [ ] Achei os dois comentários escondidos e o `generator`
- [ ] Consultei o `robots.txt` antes de baixar uma página pública
- [ ] Fiz o desafio S
- [ ] Anotei no meu [`DIARIO.md`](https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias/blob/main/DIARIO.md)

---

🎉 Terminou? [Registre sua entrega do Dia 10](https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias/issues/new?template=entreguei-o-dia.yml) · Baseado em [cipher387/python-for-OSINT-21-days](https://github.com/cipher387/python-for-OSINT-21-days) (MIT) · Adaptação pt-BR: [Ridd1kulusC0d3r/python-para-osint-21-dias](https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias)